<a href="https://colab.research.google.com/github/anandhanks20-dev/Anandhan-new/blob/main/exit_exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
#data loading
df = pd.read_excel('/content/partpdf_1772777095672_partpdf_1763620554909_eurovision_1998 to 2012.xlsx')

In [ ]:
#finding missing values
df.isnull().sum()

#analytical knowledge
by finding missing values for if the data is symmetric or random.

#correlation heat map

In [ ]:


#

# Step 1: Select only numeric audio features (exclude non-audio columns)
numeric_features = df.select_dtypes(include='number')
# If you know the audio feature column names, you can list them:
# audio_columns = ['feature1', 'feature2', 'feature3', ...]
# numeric_features = df[audio_columns]

# Step 3: Compute correlation matrix
corr_matrix = numeric_features.corr()

# Step 4: Plot correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Audio Feature Correlation Matrix Heatmap')
plt.show()

#analytical problem
in this heatmap








### **Features Chosen to Keep**
- **tempo**  
- **spectral_centroid**
- **zero_crossing_rate**
- **chroma_stft**
- **rms**

### **Features Chosen to Discard**
- **loudness**  
- **energy**  
- **spectral_rolloff**  
- **mfcc_mean**

#exploretary data analysis


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['danceability'], df['Points'], c='blue', alpha=0.5)
plt.xlabel('Danceability')
plt.ylabel('Points')
plt.title('Relationship between Danceability and Points')
plt.grid(True)
plt.show()

#analytical problem
## Relationship Between Danceability and Points

Based on the scatter plot of danceability versus points, there appears to be a **positive trend**: songs with higher danceability tend to receive more points. The points increase as danceability values rise, suggesting a preference for danceable songs among voters or listeners.

### **Hypothesis**
*"Songs with higher danceability are more likely to receive higher points, indicating that rhythm and groove play a significant role in audience appeal and scoring."*

#model training and evaluation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
import joblib # Import joblib for saving the model

# Use only selected features and target
# Updated selected_features to include existing audio features from df
selected_features = ['tempo', 'duration', 'acousticness', 'danceability', 'speechiness', 'key', 'liveness', 'time_signature', 'mode', 'valence', 'Happiness']
X = df[selected_features]
y = df['Points']

# Handle missing values (simple imputation as example)
X = X.fillna(X.mean())
y = y.fillna(y.mean())

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Gradient Boosting Regressor
gbr = GradientBoostingRegressor(random_state=42)
gbr.fit(X_train, y_train)

# Save the trained model
joblib.dump(gbr, 'gradient_boosting_model.joblib')

# Predict and Evaluate
y_pred = gbr.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")

#analaytical problem

## Mean Absolute Error (MAE) Interpretation in Eurovision Context

The Mean Absolute Error (MAE) score represents the average absolute difference between predicted and actual points for each song in the Eurovision contest.

**Your model's MAE: 44.63**

### **Contextual Meaning**
- On average, the model's predictions for Eurovision song points are off by about 44.63 points per song.
- Example: If a song actually earned 120 points and the model predicted 75.4, the absolute error would be 44.6 — close to the average error found across all test songs.

#flask application backend

In [ ]:
from flask import Flask, request, render_template
import joblib
import numpy as np

app = Flask(__name__)

# Load the trained model
# ERROR: This line is attempting to load an Excel file, not a trained model.
# You need to save the 'gbr' model (trained in a previous cell) using joblib.dump() first.
# For example, after training: joblib.dump(gbr, 'gradient_boosting_model.joblib')
# Then, load it here:
model = joblib.load('gradient_boosting_model.joblib')
# Temporarily commenting out to prevent the error until the model is saved.
# For demonstration purposes, if you were to deploy this, you would uncomment the line above
# and ensure 'gradient_boosting_model.joblib' exists.
# model = None # Placeholder, as the actual model file is not yet saved and loaded.

# Selected features for prediction - MUST match the features used during training
selected_features = ['tempo', 'duration', 'acousticness', 'danceability', 'speechiness', 'key', 'liveness', 'time_signature', 'mode', 'valence', 'Happiness']

@app.route('/')
def home():
    return render_template('index.html', prediction_text='')

@app.route('/predict', methods=['POST'])
def predict():
    if model is None:
        return render_template('index.html', prediction_text="Error: Model not loaded. Please ensure the model is saved and the path is correct.")

    try:
        # Retrieve form data for features
        input_data = []
        for feature in selected_features:
            value = request.form.get(feature) # Use .get() to avoid KeyError if a feature is missing
            # Convert to float and handle missing/invalid input
            try:
                value = float(value)
            except (ValueError, TypeError):
                value = np.nan # Keep as NaN for now
            input_data.append(value)

        # Handle missing values (mean imputation as in training)
        # For a production application, you should save the means from your training data
        # and use them here. For this example, we'll impute with 0 if NaN, as a simple placeholder.
        input_data_processed = [val if not np.isnan(val) else 0. for val in input_data]

        # Predict
        pred = model.predict([input_data_processed])[0]
        prediction_text = f"Predicted Eurovision Points: {pred:.2f}"

    except Exception as e:
        prediction_text = f"Error occurred during prediction: {e}"
        print(f"Prediction error: {e}") # For server-side debugging

    return render_template('index.html', prediction_text=prediction_text)

if __name__ == "__main__":
    # Note: Running Flask apps directly in Colab often requires tunneling solutions (like ngrok)
    # to make them accessible from a browser, or deploying to a web server.
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)


#flask application frontend

In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Eurovision Points Prediction</title>
    <style>
        body { font-family: Arial, sans-serif; background: #f7f7f7; }
        .container { max-width: 400px; margin: 40px auto; background: #fff; padding: 24px;
            border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.07); }
        form div { margin-bottom: 12px; }
        label { display: block; font-weight: bold; margin-bottom: 4px; }
        input[type="number"], input[type="text"] { width: 100%; padding: 6px 8px; border-radius: 4px; border: 1px solid #ccc; }
        button { width: 100%; padding: 10px; background: #3f51b5; color: #fff; border: none; border-radius: 4px; font-size: 1rem; cursor: pointer; }
        .result { margin-top: 20px; padding: 10px; background: #e3f2fd; border-radius: 4px; color: #333; font-size: 1.1rem; }
    </style>
</head>
<body>
    <div class="container">
        <h2>Eurovision Points Prediction</h2>
        <form method="POST" action="/predict">
            <div>
                <label for="tempo">Tempo</label>
                <input type="number" step="any" name="tempo" id="tempo" required>
            </div>
            <div>
                <label for="duration">Duration</label>
                <input type="number" step="any" name="duration" id="duration" required>
            </div>
            <div>
                <label for="acousticness">Acousticness</label>
                <input type="number" step="any" name="acousticness" id="acousticness" required>
            </div>
            <div>
                <label for="danceability">Danceability</label>
                <input type="number" step="any" name="danceability" id="danceability" required>
            </div>
            <div>
                <label for="speechiness">Speechiness</label>
                <input type="number" step="any" name="speechiness" id="speechiness" required>
            </div>
            <div>
                <label for="key">Key</label>
                <input type="number" step="any" name="key" id="key" required>
            </div>
            <div>
                <label for="liveness">Liveness</label>
                <input type="number" step="any" name="liveness" id="liveness" required>
            </div>
            <div>
                <label for="time_signature">Time Signature</label>
                <input type="number" step="any" name="time_signature" id="time_signature" required>
            </div>
            <div>
                <label for="mode">Mode</label>
                <input type="number" step="any" name="mode" id="mode" required>
            </div>
            <div>
                <label for="valence">Valence</label>
                <input type="number" step="any" name="valence" id="valence" required>
            </div>
            <div>
                <label for="Happiness">Happiness</label>
                <input type="number" step="any" name="Happiness" id="Happiness" required>
            </div>
            <button type="submit">Predict</button>
        </form>
        {% if prediction_text %}
        <div class="result">{{ prediction_text }}</div>
        {% endif %}
    </div>
</body>
</html>